In [2]:
import pandas as pd
import os
import json
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import re
import numpy as np

- recording id: A unique identifier for each recording, generated by Pupil Cloud.
- werare_name: A name manually assigned to a participant. Note that two different recordings might share the same wearer name.
- door_sequence: Represents the participant's choice of doors 1 to 6, indicating the overall path taken through the museum during the recording.
- matter_duration: The duration of time each participant spent in the matter cube.
- life_duration: The duration of time each participant spent in the life cube.
- thought_duration: The duration of time each participant spent in the thought cube.
- p1_duration :The time each participant spent engaging with panel 1.
…
- p7_duration: The time each participant spent engaging with panel 7
- phone_pickups: The count of instances where the participant picked up the phone.
- phone_duration: The overall duration of time spent on the phone.
- recording_duration: The overall duration of the recording.
- weekday



- Number of unique recording ids: 193
- Number of unique wearer names: 186
- Number of corrupted recordings: 37

# Data Reading 
wearer names edited

In [3]:
# # read data and extract wearer names from json file

# data_folder = 'eye-tracking data'
# wearer_names = {}

# folders = [folder for folder in os.listdir(data_folder) if os.path.isdir(os.path.join(data_folder, folder))]
# for folder in folders:
#     info_json_path = os.path.join(data_folder, folder, 'info.json')

#     if os.path.exists(info_json_path):
#         with open(info_json_path, 'r') as f:
#             info_data = json.load(f)
#         wearer_name = info_data['wearer_name']
#         wearer_names[folder] = wearer_name

# dfs = []

# for folder in folders:
#     events_csv_path = os.path.join(data_folder, folder, 'events.csv')

#     if os.path.exists(events_csv_path):
#         df = pd.read_csv(events_csv_path)
#         wearer_name = wearer_names[folder]
#         df['wearer_name'] = wearer_name
#         dfs.append(df)

# df = pd.concat(dfs, ignore_index=True)

# df

In [4]:
# # NO NEED TO RUN THIS CELL EVERY TIME
# # when you run this cell an input window will pop up and you can insert the correct for of mistaken wearer names in each
# # I have already replaced each incorrect wearer name with the correct form and saved the df in a csv file in the next cell

# pattern = re.compile(r'^D\d{2}-P\d{3}$')

# invalid_wearer_names = df[~df['wearer_name'].str.contains(pattern, na=False)]['wearer_name'].unique().tolist()

# print(invalid_wearer_names)

# replacement_dict = {}

# for invalid_name in invalid_wearer_names:
#     replacement = input(f"Enter replacement for {invalid_name}: ")
#     replacement_dict[invalid_name] = replacement

# df['wearer_name'] = df['wearer_name'].replace(replacement_dict)

# df


In [5]:
# df.to_csv('wearer_names_edited.csv', index=False)
df = pd.read_csv('wearer_names_edited.csv')

In [6]:
df

,recording id,timestamp [ns],name,type,wearer_name
0,b5cff087-0b53-499e-997d-05af39002d55,1695733480308000000,recording.begin,recording,D01-P010
1,b5cff087-0b53-499e-997d-05af39002d55,1695733580411684000,exterior21.begin,cloud,D01-P010
2,b5cff087-0b53-499e-997d-05af39002d55,1695733640648304000,exterior21.end,cloud,D01-P010
3,b5cff087-0b53-499e-997d-05af39002d55,1695733664728872000,matter1.begin,cloud,D01-P010
4,b5cff087-0b53-499e-997d-05af39002d55,1695733664728872000,d1.in,cloud,D01-P010
...,...,...,...,...,...
4169,adb20035-13c3-41f1-b08d-07c0f338a2a6,1696192433681506000,d6.out,cloud,D01-P064
4170,adb20035-13c3-41f1-b08d-07c0f338a2a6,1696192435749759000,thought1.end,cloud,D01-P064
4171,adb20035-13c3-41f1-b08d-07c0f338a2a6,1696192436945640000,p71.begin,cloud,D01-P064
4172,adb20035-13c3-41f1-b08d-07c0f338a2a6,1696192472771558000,p71.end,cloud,D01-P064


In [7]:
# how many unique recrding? how many unique wearers?

unique_recording_ids = df['recording id'].nunique()
print("Number of unique recording ids:", unique_recording_ids)

unique_wearer_names = df['wearer_name'].nunique()
print("Number of unique wearer names:", unique_wearer_names)

Number of unique recording ids: 193
Number of unique wearer names: 186


In [8]:
wearer_names_with_multiple_ids = df.groupby('wearer_name').filter(lambda x: x['recording id'].nunique() > 1)['wearer_name'].unique().tolist()
print("Wearer names with more than one recording id:", wearer_names_with_multiple_ids)

Wearer names with more than one recording id: ['D05-P019', 'D05-P025', 'D03-P040', 'D02-P041', 'D03-P059', 'D04-P044', 'D01-P061']


In [9]:
# Has anyone not passed through any of the doors?
# interrupted video: 'D02-P022', 'D03-P033', 'D03-P037', 'D01-P036', 'D04-P035', 
# gone the wrong way: 'D04-P038'

filtered_df = df[df['name'].str.startswith('d', na=False)]
wearer_names_with_d = filtered_df['wearer_name'].unique()
filtered_df_2 = df[~df['wearer_name'].isin(wearer_names_with_d)]
wearer_names_without_d = filtered_df_2['wearer_name'].unique()
wearer_names_without_d

array(['D02-P022', 'D03-P033', 'D03-P037', 'D01-P036', 'D04-P035',
       'D04-P038'], dtype=object)

In [10]:
# label corrupted video by setting has_corruption to True. We will identify more of these by checking the vieos of suspecious recordings. 

df['has_corruption'] = False
wearer_names_to_set_true = ['D02-P022', 'D03-P033', 'D03-P037', 'D01-P036', 'D04-P035']
df.loc[df['wearer_name'].isin(wearer_names_to_set_true), 'has_corruption'] = True


# Door Sequence

In [11]:
# Door sequences have been crafted as a new feature.

doors_df = df[df['name'].str.startswith('d')].copy()
doors_df['door_sequence'] = doors_df.groupby('recording id')['name'].transform(lambda x: ','.join(x))
doors_df = doors_df.drop_duplicates(subset=['recording id'], keep='first')
doors_df = doors_df.drop(columns=['name', 'type', 'timestamp [ns]'])
doors_df

,recording id,wearer_name,has_corruption,door_sequence
4,b5cff087-0b53-499e-997d-05af39002d55,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out"
18,0ca2b022-aee7-4299-a2a1-f70663e48628,D03-P011,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d6.out"
50,5459d7cb-ad76-41b5-8547-78d0993ce78b,D03-P012,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out"
84,d954354a-0daf-434e-b34d-fba86285e06d,D01-P011,False,"d1.in,d2.out,d3.in,d3.out,d3.in,d4.out,d6.in,d..."
118,b726709d-b664-48c0-8161-0b5b8533d84b,D05-P013,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d5.out"
...,...,...,...,...
4039,40548af0-64d4-4e4c-baa8-f0f8c76ac823,D03-P066,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out"
4084,a338fabd-600c-4be0-88a4-a00b19e3b487,D01-P063,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out"
4102,8de10704-68d9-4bdd-957c-42a60fdbf562,D05-P050,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out"
4119,6c8a0694-0c8f-4ae3-978e-e3657f3f4fd1,D02-P058,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d6.out"


In [12]:
# Combine sequences from wearers with two or more recordings.

doors_df = doors_df.groupby('wearer_name')['door_sequence'].agg(lambda x: ','.join(x)).reset_index()
doors_df

,wearer_name,door_sequence
0,D01-P010,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out"
1,D01-P011,"d1.in,d2.out,d3.in,d3.out,d3.in,d4.out,d6.in,d..."
2,D01-P012,"d1.in,d2.out,d3.in,d4.out,d3.in,d4.out,d5.in,d..."
3,D01-P013,"d1.in,d2.out,d3.in,d4.out"
4,D01-P014,"d1.in,d2.out"
...,...,...
175,D05-P046,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out"
176,D05-P047,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out"
177,D05-P048,"d5.in,d6.out"
178,D05-P049,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out"


In [13]:
# Sequences with four or fewer door labels are considered suspicious, indicating a potential interruption in the video.

num_doors = doors_df['door_sequence'].str.split(',').apply(len)
filtered_wearer_names = doors_df.loc[num_doors <= 4, ['wearer_name', 'door_sequence']]
print(len(filtered_wearer_names))
filtered_wearer_names

46


,wearer_name,door_sequence
3,D01-P013,"d1.in,d2.out,d3.in,d4.out"
4,D01-P014,"d1.in,d2.out"
6,D01-P028,"d1.in,d2.out,d3.in,d4.out"
7,D01-P029,"d3.in,d4.out,d5.in,d6.out"
8,D01-P030,d1.in
13,D01-P037,"d3.in,d4.out,d5.in,d6.out"
16,D01-P040,"d3.in,d4.out,d5.in,d6.out"
25,D01-P049,"d1.in,d2.out,d3.in,d4.out"
32,D01-P058,"d1.in,d2.out,d3.in"
33,D01-P059,"d3.in,d4.out,d5.in,d6.out"


In [14]:
# After reviewing the videos, it was determined that these recordings are corrupted.

wearer_names_to_set_true = ['D05-P035', 'D04-P045', 'D04-P044', 'D04-P040', 'D04-P031', 'D03-P058', 'D03-P057', 'D03-P054', 'D03-P053', 'D03-P041', 'D03-P040', 'D03-P039', 'D03-P038', 'D03-P035', 'D03-P034', 'D02-P018', 'D02-P025', 'D02-P047', 'D02-P040', 'D01-P014','D01-P028', 'D01-P030', 'D02-P037', 'D01-P062', 'D01-P058']
df.loc[df['wearer_name'].isin(wearer_names_to_set_true), 'has_corruption'] = True


In [15]:
corrupted_participant_recordings = len(df[df['has_corruption']].groupby('wearer_name'))
print("Number of corrupted recordings:", corrupted_participant_recordings)

Number of corrupted recordings: 30


In [16]:
df = pd.merge(df, doors_df, on='wearer_name', how='outer')



In [17]:
df

,recording id,timestamp [ns],name,type,wearer_name,has_corruption,door_sequence
0,b5cff087-0b53-499e-997d-05af39002d55,1695733480308000000,recording.begin,recording,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out"
1,b5cff087-0b53-499e-997d-05af39002d55,1695733580411684000,exterior21.begin,cloud,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out"
2,b5cff087-0b53-499e-997d-05af39002d55,1695733640648304000,exterior21.end,cloud,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out"
3,b5cff087-0b53-499e-997d-05af39002d55,1695733664728872000,matter1.begin,cloud,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out"
4,b5cff087-0b53-499e-997d-05af39002d55,1695733664728872000,d1.in,cloud,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out"
...,...,...,...,...,...,...,...
4169,adb20035-13c3-41f1-b08d-07c0f338a2a6,1696192433681506000,d6.out,cloud,D01-P064,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d6.out"
4170,adb20035-13c3-41f1-b08d-07c0f338a2a6,1696192435749759000,thought1.end,cloud,D01-P064,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d6.out"
4171,adb20035-13c3-41f1-b08d-07c0f338a2a6,1696192436945640000,p71.begin,cloud,D01-P064,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d6.out"
4172,adb20035-13c3-41f1-b08d-07c0f338a2a6,1696192472771558000,p71.end,cloud,D01-P064,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d6.out"


In [18]:
wearer_names_to_set_true = ['D01-P056', 'D02-P052', 'D05-P043', 'D03-P027', 'D03-P036', 'D04-P029', 'D04-P025']
df.loc[df['wearer_name'].isin(wearer_names_to_set_true), 'has_corruption'] = True

In [19]:
# Check for mistakes, which may result from errors in labeling or video interruptions. Mistakes include:

# 1. An entrance label should not be followed by another entrance label.
# 2. An exit label should not be followed by another exit label.
# 3. The sequence should not end with an entrance label.

def check_sequence(sequence):
    if pd.isna(sequence):
        return True  # Assuming NaN is not incorrect
    events = sequence.split(',')
    for i in range(len(events) - 1):
        if (events[i].endswith('in') and events[i + 1].endswith('in')) or \
           (events[i].endswith('out') and events[i + 1].endswith('out')):
            return True
    return not events[-1].endswith('in')

filtered_df = df[df['has_corruption'] == False]

mistakes_df = filtered_df[~filtered_df['door_sequence'].apply(lambda x: check_sequence(x))]

grouped_mistakes_df = mistakes_df.groupby('wearer_name').agg({
    'door_sequence': 'first',
    'has_corruption': 'first'
})

grouped_mistakes_df

,door_sequence,has_corruption
wearer_name,,


# Items Duration

matter

In [20]:
matter_df = df[df['name'].str.startswith('matter', na=False)].copy()

unique_wearer_names_matter_begin = df[df['name'] == 'matter1.begin']['wearer_name'].nunique()
print("Number of unique wearer names with matter1.begin:", unique_wearer_names_matter_begin)

unique_wearer_names_matter_end = df[df['name'] == 'matter1.end']['wearer_name'].nunique()
print("Number of unique wearer names with matter1.end:", unique_wearer_names_matter_end)

ratio_matter_begin = unique_wearer_names_matter_begin / unique_wearer_names

print("Ratio of unique wearer names with matter1.begin to all wearer names:", ratio_matter_begin)

Number of unique wearer names with matter1.begin: 167
Number of unique wearer names with matter1.end: 167
Ratio of unique wearer names with matter1.begin to all wearer names: 0.8978494623655914


In [21]:
matter_df = matter_df.drop_duplicates(subset=['wearer_name', 'name'])

In [22]:
matter_df['timestamp [ns]'] = pd.to_datetime(matter_df['timestamp [ns]'], unit='ns')
pivot_matter_df = matter_df.pivot(index=['wearer_name'], columns='name', values='timestamp [ns]')

duration_matter1 = pivot_matter_df['matter1.end'] - pivot_matter_df['matter1.begin']
duration_matter2 = pivot_matter_df['matter2.end'] - pivot_matter_df['matter2.begin']

duration_matter1 = pd.to_timedelta(duration_matter1.fillna(0))
duration_matter2 = pd.to_timedelta(duration_matter2.fillna(0))

pivot_matter_df['matter_duration'] = duration_matter1 + duration_matter2
pivot_matter_df = pivot_matter_df.reset_index()
pivot_matter_df


name,wearer_name,matter1.begin,matter1.end,matter2.begin,matter2.end,matter_duration
0,D01-P010,2023-09-26 13:07:44.728872,2023-09-26 13:09:42.595494,NaT,NaT,0 days 00:01:57.866622
1,D01-P011,2023-09-26 13:36:32.518274,2023-09-26 13:37:53.825712,NaT,NaT,0 days 00:01:21.307438
2,D01-P012,2023-09-26 13:59:45.585248,2023-09-26 14:01:37.916628,NaT,NaT,0 days 00:01:52.331380
3,D01-P013,2023-09-26 14:45:07.344004,2023-09-26 14:46:49.696504,NaT,NaT,0 days 00:01:42.352500
4,D01-P014,2023-09-26 15:21:49.255122,2023-09-26 15:22:32.073762,NaT,NaT,0 days 00:00:42.818640
...,...,...,...,...,...,...
162,D05-P045,2023-10-01 14:07:02.593094,2023-10-01 14:08:58.399748,NaT,NaT,0 days 00:01:55.806654
163,D05-P046,2023-10-01 14:35:04.186740,2023-10-01 14:37:53.848529,NaT,NaT,0 days 00:02:49.661789
164,D05-P047,2023-10-01 17:19:07.512440,2023-10-01 17:22:00.507727,NaT,NaT,0 days 00:02:52.995287
165,D05-P049,2023-10-01 19:35:00.186965,2023-10-01 19:36:26.887644,NaT,NaT,0 days 00:01:26.700679


In [23]:
pivot_matter_df

name,wearer_name,matter1.begin,matter1.end,matter2.begin,matter2.end,matter_duration
0,D01-P010,2023-09-26 13:07:44.728872,2023-09-26 13:09:42.595494,NaT,NaT,0 days 00:01:57.866622
1,D01-P011,2023-09-26 13:36:32.518274,2023-09-26 13:37:53.825712,NaT,NaT,0 days 00:01:21.307438
2,D01-P012,2023-09-26 13:59:45.585248,2023-09-26 14:01:37.916628,NaT,NaT,0 days 00:01:52.331380
3,D01-P013,2023-09-26 14:45:07.344004,2023-09-26 14:46:49.696504,NaT,NaT,0 days 00:01:42.352500
4,D01-P014,2023-09-26 15:21:49.255122,2023-09-26 15:22:32.073762,NaT,NaT,0 days 00:00:42.818640
...,...,...,...,...,...,...
162,D05-P045,2023-10-01 14:07:02.593094,2023-10-01 14:08:58.399748,NaT,NaT,0 days 00:01:55.806654
163,D05-P046,2023-10-01 14:35:04.186740,2023-10-01 14:37:53.848529,NaT,NaT,0 days 00:02:49.661789
164,D05-P047,2023-10-01 17:19:07.512440,2023-10-01 17:22:00.507727,NaT,NaT,0 days 00:02:52.995287
165,D05-P049,2023-10-01 19:35:00.186965,2023-10-01 19:36:26.887644,NaT,NaT,0 days 00:01:26.700679


In [24]:
df = pd.merge(df, pivot_matter_df[['wearer_name', 'matter_duration']], on='wearer_name', how='outer')

life

In [25]:
life_df = df[df['name'].str.startswith('life', na=False)].copy()

unique_wearer_names_life_begin = df[df['name'] == 'life1.begin']['wearer_name'].nunique()
print("Number of unique wearer names with life1.begin:", unique_wearer_names_life_begin)

unique_wearer_names_life_end = df[df['name'] == 'life1.end']['wearer_name'].nunique()
print("Number of unique wearer names with life1.end:", unique_wearer_names_life_end)

ratio_life_begin = unique_wearer_names_life_begin / unique_wearer_names

print("Ratio of unique wearer names with life1.begin to all wearer names:", ratio_life_begin)

Number of unique wearer names with life1.begin: 171
Number of unique wearer names with life1.end: 171
Ratio of unique wearer names with life1.begin to all wearer names: 0.9193548387096774


In [26]:
life_df = life_df.groupby(['wearer_name', 'recording id', 'name'])['timestamp [ns]'].agg('first').reset_index()
pivot_life_df = life_df.pivot(index=['wearer_name', 'recording id'], columns='name', values='timestamp [ns]')


life_df['timestamp [ns]'] = pd.to_datetime(life_df['timestamp [ns]'], unit='ns')
pivot_life_df = life_df.pivot(index=['wearer_name', 'recording id'], columns='name', values='timestamp [ns]')
pivot_life_df['life_duration'] = pivot_life_df['life1.end'] - pivot_life_df['life1.begin']
pivot_life_df = pivot_life_df.reset_index()

In [27]:
df = pd.merge(df, pivot_life_df[['wearer_name', 'life_duration']], on='wearer_name', how='outer')

thought

In [28]:
thought_df = df[df['name'].str.startswith('thought', na=False)].copy()

unique_wearer_names_thought_begin = df[df['name'] == 'thought1.begin']['wearer_name'].nunique()
print("Number of unique wearer names with thought1.begin:", unique_wearer_names_thought_begin)
unique_wearer_names_thought_end = df[df['name'] == 'thought1.end']['wearer_name'].nunique()
print("Number of unique wearer names with thought1.end:", unique_wearer_names_thought_end)

ratio_thought_begin = unique_wearer_names_thought_begin / unique_wearer_names

print("Ratio of unique wearer names with life1.begin to all wearer names:", ratio_thought_begin)

Number of unique wearer names with thought1.begin: 144
Number of unique wearer names with thought1.end: 144
Ratio of unique wearer names with life1.begin to all wearer names: 0.7741935483870968


In [29]:
thought_df = thought_df.drop_duplicates(subset=['wearer_name', 'name'])

In [30]:
thought_df['timestamp [ns]'] = pd.to_datetime(thought_df['timestamp [ns]'], unit='ns')
pivot_thought_df = thought_df.pivot(index=['wearer_name', 'recording id'], columns='name', values='timestamp [ns]')
pivot_thought_df['thought_duration'] = pivot_thought_df['thought1.end'] - pivot_thought_df['thought1.begin']
pivot_thought_df = pivot_thought_df.reset_index()

In [31]:
df = pd.merge(df, pivot_thought_df[['wearer_name', 'thought_duration']], on='wearer_name', how='outer')

In [32]:
prefixes = ['p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7']

result_df = pd.DataFrame()

for prefix in prefixes:
    current_df = df[df['name'].str.startswith(prefix, na=False)].copy()

    current_df = current_df.drop_duplicates(subset=['wearer_name', 'name'])

    unique_wearer_names_begin = df[df['name'] == f'{prefix}1.begin']['wearer_name'].nunique()
    unique_wearer_names_end = df[df['name'] == f'{prefix}1.end']['wearer_name'].nunique()

    ratio = unique_wearer_names_begin / unique_wearer_names

    print(f"Prefix: {prefix}")
    print("Number of unique wearer names with {prefix}1.begin:", unique_wearer_names_begin)
    print("Number of unique wearer names with {prefix}1.end:", unique_wearer_names_end)
    print("Ratio:", ratio)


Prefix: p1
Number of unique wearer names with {prefix}1.begin: 64
Number of unique wearer names with {prefix}1.end: 64
Ratio: 0.34408602150537637
Prefix: p2
Number of unique wearer names with {prefix}1.begin: 55
Number of unique wearer names with {prefix}1.end: 55
Ratio: 0.2956989247311828
Prefix: p3
Number of unique wearer names with {prefix}1.begin: 36
Number of unique wearer names with {prefix}1.end: 36
Ratio: 0.1935483870967742
Prefix: p4
Number of unique wearer names with {prefix}1.begin: 57
Number of unique wearer names with {prefix}1.end: 57
Ratio: 0.3064516129032258
Prefix: p5
Number of unique wearer names with {prefix}1.begin: 31
Number of unique wearer names with {prefix}1.end: 31
Ratio: 0.16666666666666666
Prefix: p6
Number of unique wearer names with {prefix}1.begin: 19
Number of unique wearer names with {prefix}1.end: 19
Ratio: 0.10215053763440861
Prefix: p7
Number of unique wearer names with {prefix}1.begin: 25
Number of unique wearer names with {prefix}1.end: 25
Ratio: 0

In [33]:
p2_df = df[df['name'].str.startswith('p2', na=False)].copy()
p2_df = p2_df.drop_duplicates(subset=['wearer_name', 'name'])
p2_df['timestamp [ns]'] = pd.to_datetime(p2_df['timestamp [ns]'], unit='ns')
pivot_p2_df = p2_df.pivot(index=['wearer_name', 'recording id'], columns='name', values='timestamp [ns]')
pivot_p2_df['p2_duration'] = pivot_p2_df['p21.end'] - pivot_p2_df['p21.begin']
p2 = pivot_p2_df.reset_index()
df = pd.merge(df, p2[['wearer_name', 'p2_duration']], on='wearer_name', how='outer')


In [34]:
df

,recording id,timestamp [ns],name,type,wearer_name,has_corruption,door_sequence,matter_duration,life_duration,thought_duration,p2_duration
0,b5cff087-0b53-499e-997d-05af39002d55,1695733480308000000,recording.begin,recording,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out",0 days 00:01:57.866622,0 days 00:01:14.397229,0 days 00:12:32.545630,NaT
1,b5cff087-0b53-499e-997d-05af39002d55,1695733580411684000,exterior21.begin,cloud,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out",0 days 00:01:57.866622,0 days 00:01:14.397229,0 days 00:12:32.545630,NaT
2,b5cff087-0b53-499e-997d-05af39002d55,1695733640648304000,exterior21.end,cloud,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out",0 days 00:01:57.866622,0 days 00:01:14.397229,0 days 00:12:32.545630,NaT
3,b5cff087-0b53-499e-997d-05af39002d55,1695733664728872000,matter1.begin,cloud,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out",0 days 00:01:57.866622,0 days 00:01:14.397229,0 days 00:12:32.545630,NaT
4,b5cff087-0b53-499e-997d-05af39002d55,1695733664728872000,d1.in,cloud,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out",0 days 00:01:57.866622,0 days 00:01:14.397229,0 days 00:12:32.545630,NaT
...,...,...,...,...,...,...,...,...,...,...,...
4237,adb20035-13c3-41f1-b08d-07c0f338a2a6,1696192433681506000,d6.out,cloud,D01-P064,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d6.out",0 days 00:01:08.544715,0 days 00:05:27.599071,0 days 00:06:14.862188,0 days 00:00:32.971770
4238,adb20035-13c3-41f1-b08d-07c0f338a2a6,1696192435749759000,thought1.end,cloud,D01-P064,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d6.out",0 days 00:01:08.544715,0 days 00:05:27.599071,0 days 00:06:14.862188,0 days 00:00:32.971770
4239,adb20035-13c3-41f1-b08d-07c0f338a2a6,1696192436945640000,p71.begin,cloud,D01-P064,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d6.out",0 days 00:01:08.544715,0 days 00:05:27.599071,0 days 00:06:14.862188,0 days 00:00:32.971770
4240,adb20035-13c3-41f1-b08d-07c0f338a2a6,1696192472771558000,p71.end,cloud,D01-P064,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d6.out",0 days 00:01:08.544715,0 days 00:05:27.599071,0 days 00:06:14.862188,0 days 00:00:32.971770


In [35]:
df = pd.read_csv('sequence_and_item_duration.csv')

In [36]:
df

,Unnamed: 0,recording id,timestamp [ns],name,type,wearer_name,has_corruption,door_sequence,matter_duration,life_duration,thought_duration,p1_duration,p2_duration,p3_duration,p4_duration,p5_duration,p6_duration,p7_duration
0,0,b5cff087-0b53-499e-997d-05af39002d55,1695733480308000000,recording.begin,recording,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out",0 days 00:01:57.866622,0 days 00:01:14.397229,0 days 00:12:32.545630,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,b5cff087-0b53-499e-997d-05af39002d55,1695733580411684000,exterior21.begin,cloud,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out",0 days 00:01:57.866622,0 days 00:01:14.397229,0 days 00:12:32.545630,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,b5cff087-0b53-499e-997d-05af39002d55,1695733640648304000,exterior21.end,cloud,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out",0 days 00:01:57.866622,0 days 00:01:14.397229,0 days 00:12:32.545630,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,b5cff087-0b53-499e-997d-05af39002d55,1695733664728872000,matter1.begin,cloud,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out",0 days 00:01:57.866622,0 days 00:01:14.397229,0 days 00:12:32.545630,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,b5cff087-0b53-499e-997d-05af39002d55,1695733664728872000,d1.in,cloud,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out",0 days 00:01:57.866622,0 days 00:01:14.397229,0 days 00:12:32.545630,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4237,4237,adb20035-13c3-41f1-b08d-07c0f338a2a6,1696192433681506000,d6.out,cloud,D01-P064,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d6.out",0 days 00:01:08.544715,0 days 00:05:27.599071,0 days 00:06:14.862188,0 days 00:02:01.244742,0 days 00:00:32.971770,NaN,NaN,NaN,NaN,0 days 00:00:35.825918
4238,4238,adb20035-13c3-41f1-b08d-07c0f338a2a6,1696192435749759000,thought1.end,cloud,D01-P064,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d6.out",0 days 00:01:08.544715,0 days 00:05:27.599071,0 days 00:06:14.862188,0 days 00:02:01.244742,0 days 00:00:32.971770,NaN,NaN,NaN,NaN,0 days 00:00:35.825918
4239,4239,adb20035-13c3-41f1-b08d-07c0f338a2a6,1696192436945640000,p71.begin,cloud,D01-P064,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d6.out",0 days 00:01:08.544715,0 days 00:05:27.599071,0 days 00:06:14.862188,0 days 00:02:01.244742,0 days 00:00:32.971770,NaN,NaN,NaN,NaN,0 days 00:00:35.825918
4240,4240,adb20035-13c3-41f1-b08d-07c0f338a2a6,1696192472771558000,p71.end,cloud,D01-P064,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d6.out",0 days 00:01:08.544715,0 days 00:05:27.599071,0 days 00:06:14.862188,0 days 00:02:01.244742,0 days 00:00:32.971770,NaN,NaN,NaN,NaN,0 days 00:00:35.825918


# Phone Engagement

In [37]:
phone_df = df[df['name'].str.startswith('phone', na=False)].copy()
phone_df['timestamp [ns]'] = pd.to_datetime(phone_df['timestamp [ns]'])
phone_df = phone_df.sort_values(by=['wearer_name', 'timestamp [ns]'])

unique_wearer_names_phone_pickup = df[df['name'] == 'phone.pickup']['wearer_name'].nunique()
print(unique_wearer_names_phone_pickup)
unique_wearer_names_phone_putdown = df[df['name'] == 'phone.putdown']['wearer_name'].nunique()
print(unique_wearer_names_phone_putdown)

130
130


In [38]:
grouped_phone_df = phone_df.groupby('wearer_name')

def custom_aggregation(group):
    phone_pickups_count = group['name'].str.count('phone.pickup').sum()
    time_diffs = group['timestamp [ns]'].diff().iloc[1::2].sum()
    return pd.Series({'phone_pickups': phone_pickups_count, 'phone_duration': time_diffs})

phone_df = grouped_phone_df.apply(custom_aggregation).reset_index()


In [39]:
phone_df

,wearer_name,phone_pickups,phone_duration
0,D01-P011,5,0 days 00:01:35.651010
1,D01-P012,2,0 days 00:02:12.876348
2,D01-P013,1,0 days 00:01:48.432044
3,D01-P014,3,0 days 00:00:10.240140
4,D01-P015,5,0 days 00:01:40.857380
...,...,...,...
125,D05-P042,3,0 days 00:02:25.810939
126,D05-P043,5,0 days 00:03:00.642784
127,D05-P047,1,0 days 00:00:12.086640
128,D05-P048,3,0 days 00:00:19.918620


In [40]:
df = pd.merge(df, phone_df[['wearer_name', 'phone_pickups', 'phone_duration']], on='wearer_name', how='outer')


# Overall Duration

In [41]:
# recording_df = df[df['name'].str.startswith('recording', na=False)].copy()
# recording_df = recording_df.drop_duplicates(subset=['wearer_name', 'name'])
# recording_df['timestamp [ns]'] = pd.to_datetime(recording_df['timestamp [ns]'], unit='ns')
# pivot_recording_df = recording_df.pivot(index=['wearer_name', 'recording id'], columns='name', values='timestamp [ns]')
# pivot_recording_df['overall_duration'] = pivot_recording_df['recording.end'] - pivot_recording_df['recording.begin']
# overall_duration = pivot_recording_df.reset_index()
# df = pd.merge(df, overall_duration[['wearer_name', 'overall_duration']], on='wearer_name', how='outer')

In [42]:
recording_df = df[df['name'].str.startswith('recording', na=False)].copy()
recording_df['timestamp [ns]'] = pd.to_datetime(recording_df['timestamp [ns]'])
recording_df = recording_df.sort_values(by=['wearer_name', 'timestamp [ns]'])

grouped_recording_df = recording_df.groupby('wearer_name', as_index=False)

def custom_aggregation(group):
    # Drop duplicates within each group
    group = group.drop_duplicates(subset=['timestamp [ns]'])
    
    # Calculate time differences and handle NaN values
    time_diffs = group['timestamp [ns]'].diff()
    time_diffs = time_diffs.iloc[1::2].dropna().sum()
    
    return pd.Series({'overall_duration': time_diffs})

recording_df = grouped_recording_df.apply(custom_aggregation).reset_index(drop=True)
recording_df


,wearer_name,overall_duration
0,D01-P010,0 days 00:20:09.907000
1,D01-P011,0 days 00:19:53.243000
2,D01-P012,0 days 00:22:58.232000
3,D01-P013,0 days 00:08:26.666000
4,D01-P014,0 days 00:41:52.585000
...,...,...
181,D05-P046,0 days 00:28:18.878000
182,D05-P047,0 days 00:07:38.276000
183,D05-P048,0 days 00:21:55.838498120
184,D05-P049,0 days 00:07:08.867000


In [43]:
selected_row = recording_df[recording_df['overall_duration'] == pd.Timedelta(0)]
print(selected_row)

Empty DataFrame
Columns: [wearer_name, overall_duration]
Index: []


In [44]:
df = pd.merge(df, recording_df[['wearer_name', 'overall_duration']], on='wearer_name', how='outer')


# Week Day

In [45]:
recording_df = df[df['name'].str.startswith('recording', na=False)].copy()
recording_df['timestamp [ns]'] = pd.to_datetime(recording_df['timestamp [ns]'])
recording_df = recording_df.sort_values(by=['wearer_name', 'timestamp [ns]'])
first_timestamp_per_wearer = recording_df.groupby('wearer_name')['timestamp [ns]'].min()

weekday_per_wearer = first_timestamp_per_wearer.dt.day_name()

weekday_df = pd.DataFrame({'wearer_name': weekday_per_wearer.index, 'weekday': weekday_per_wearer.values})

weekday_df


,wearer_name,weekday
0,D01-P010,Tuesday
1,D01-P011,Tuesday
2,D01-P012,Tuesday
3,D01-P013,Tuesday
4,D01-P014,Tuesday
...,...,...
181,D05-P046,Sunday
182,D05-P047,Sunday
183,D05-P048,Sunday
184,D05-P049,Sunday


In [46]:
df = pd.merge(df, weekday_df[['wearer_name', 'weekday']], on='wearer_name', how='outer')


In [47]:
df.to_csv('eye-tracking-events-data.csv')

In [48]:
df

,Unnamed: 0,recording id,timestamp [ns],name,type,wearer_name,has_corruption,door_sequence,matter_duration,life_duration,...,p2_duration,p3_duration,p4_duration,p5_duration,p6_duration,p7_duration,phone_pickups,phone_duration,overall_duration,weekday
0,0,b5cff087-0b53-499e-997d-05af39002d55,1695733480308000000,recording.begin,recording,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out",0 days 00:01:57.866622,0 days 00:01:14.397229,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,0 days 00:20:09.907000,Tuesday
1,1,b5cff087-0b53-499e-997d-05af39002d55,1695733580411684000,exterior21.begin,cloud,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out",0 days 00:01:57.866622,0 days 00:01:14.397229,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,0 days 00:20:09.907000,Tuesday
2,2,b5cff087-0b53-499e-997d-05af39002d55,1695733640648304000,exterior21.end,cloud,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out",0 days 00:01:57.866622,0 days 00:01:14.397229,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,0 days 00:20:09.907000,Tuesday
3,3,b5cff087-0b53-499e-997d-05af39002d55,1695733664728872000,matter1.begin,cloud,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out",0 days 00:01:57.866622,0 days 00:01:14.397229,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,0 days 00:20:09.907000,Tuesday
4,4,b5cff087-0b53-499e-997d-05af39002d55,1695733664728872000,d1.in,cloud,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out",0 days 00:01:57.866622,0 days 00:01:14.397229,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,0 days 00:20:09.907000,Tuesday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4237,4237,adb20035-13c3-41f1-b08d-07c0f338a2a6,1696192433681506000,d6.out,cloud,D01-P064,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d6.out",0 days 00:01:08.544715,0 days 00:05:27.599071,...,0 days 00:00:32.971770,NaN,NaN,NaN,NaN,0 days 00:00:35.825918,4.0,0 days 00:01:54.531953,0 days 00:18:21.111000,Sunday
4238,4238,adb20035-13c3-41f1-b08d-07c0f338a2a6,1696192435749759000,thought1.end,cloud,D01-P064,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d6.out",0 days 00:01:08.544715,0 days 00:05:27.599071,...,0 days 00:00:32.971770,NaN,NaN,NaN,NaN,0 days 00:00:35.825918,4.0,0 days 00:01:54.531953,0 days 00:18:21.111000,Sunday
4239,4239,adb20035-13c3-41f1-b08d-07c0f338a2a6,1696192436945640000,p71.begin,cloud,D01-P064,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d6.out",0 days 00:01:08.544715,0 days 00:05:27.599071,...,0 days 00:00:32.971770,NaN,NaN,NaN,NaN,0 days 00:00:35.825918,4.0,0 days 00:01:54.531953,0 days 00:18:21.111000,Sunday
4240,4240,adb20035-13c3-41f1-b08d-07c0f338a2a6,1696192472771558000,p71.end,cloud,D01-P064,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d6.out",0 days 00:01:08.544715,0 days 00:05:27.599071,...,0 days 00:00:32.971770,NaN,NaN,NaN,NaN,0 days 00:00:35.825918,4.0,0 days 00:01:54.531953,0 days 00:18:21.111000,Sunday


In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4242 entries, 0 to 4241
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype          
---  ------            --------------  -----          
 0   Unnamed: 0        4242 non-null   int64          
 1   recording id      4242 non-null   object         
 2   timestamp [ns]    4242 non-null   int64          
 3   name              4242 non-null   object         
 4   type              4242 non-null   object         
 5   wearer_name       4242 non-null   object         
 6   has_corruption    4242 non-null   bool           
 7   door_sequence     4214 non-null   object         
 8   matter_duration   3918 non-null   object         
 9   life_duration     4077 non-null   object         
 10  thought_duration  3587 non-null   object         
 11  p1_duration       1558 non-null   object         
 12  p2_duration       1396 non-null   object         
 13  p3_duration       1052 non-null   object         
 14  p4_durat

In [50]:
columns_to_drop = ['Unnamed: 0', 'timestamp [ns]','name', 'type' ]
df = df.drop(columns=columns_to_drop)
df.drop_duplicates(subset=['wearer_name'], keep='first', inplace=True)
df.reset_index(drop=True, inplace=True)
columns_to_fill = ['matter_duration', 'life_duration', 'thought_duration', 'p1_duration', 'p2_duration', 'p3_duration', 'p4_duration', 'p5_duration', 'p6_duration', 'p7_duration', 'phone_pickups', 'phone_duration']
df.loc[:, columns_to_fill] = df[columns_to_fill].fillna(0)

In [51]:
df['matter_duration'] = pd.to_timedelta(df['matter_duration'])
df['life_duration'] = pd.to_timedelta(df['life_duration'])
df['thought_duration'] = pd.to_timedelta(df['thought_duration'])
df['p1_duration'] = pd.to_timedelta(df['p1_duration'])
df['p2_duration'] = pd.to_timedelta(df['p2_duration'])
df['p3_duration'] = pd.to_timedelta(df['p3_duration'])
df['p4_duration'] = pd.to_timedelta(df['p4_duration'])
df['p5_duration'] = pd.to_timedelta(df['p5_duration'])
df['p6_duration'] = pd.to_timedelta(df['p6_duration'])
df['p7_duration'] = pd.to_timedelta(df['p7_duration'])
df['phone_duration'] = pd.to_timedelta(df['phone_duration'])
df['overall_duration'] = pd.to_timedelta(df['overall_duration'])

In [52]:
df['overall_panel_duration'] = df['p1_duration'] + df['p2_duration'] + df['p3_duration'] + df['p4_duration'] + df['p5_duration'] + df['p6_duration'] + df['p7_duration']
df['number_of_panels'] = df[['p1_duration', 'p2_duration', 'p3_duration', 'p4_duration', 'p5_duration', 'p6_duration', 'p7_duration']].astype(bool).sum(axis=1)

df['matter_panel_num'] = df[['p2_duration', 'p3_duration']].astype(bool).sum(axis=1)
df['life_panel_num'] = df[['p4_duration', 'p5_duration']].astype(bool).sum(axis=1)
df['thought_panel_num'] = df[['p6_duration', 'p7_duration']].astype(bool).sum(axis=1)

df['matter_panel_duration'] =  df['p2_duration'] + df['p3_duration']
df['life_panel_duration'] = df['p4_duration'] + df['p5_duration']
df['thought_panel_duration'] = df['p6_duration'] + df['p7_duration']

In [53]:
df['matter_duration'] = df['matter_duration'].dt.total_seconds() / 60
df['life_duration'] = df['life_duration'].dt.total_seconds() / 60
df['thought_duration'] = df['thought_duration'].dt.total_seconds() / 60
df['p1_duration'] = df['p1_duration'].dt.total_seconds() / 60
df['p2_duration'] = df['p2_duration'].dt.total_seconds() / 60
df['p3_duration'] = df['p3_duration'].dt.total_seconds() / 60
df['p4_duration'] = df['p4_duration'].dt.total_seconds() / 60
df['p5_duration'] = df['p5_duration'].dt.total_seconds() / 60
df['p6_duration'] = df['p6_duration'].dt.total_seconds() / 60
df['p7_duration'] = df['p7_duration'].dt.total_seconds() / 60
df['phone_duration'] = df['phone_duration'].dt.total_seconds() / 60
df['overall_duration'] = df['overall_duration'].dt.total_seconds() / 60
df['overall_panel_duration'] = df['overall_panel_duration'].dt.total_seconds() / 60     
df['matter_panel_duration'] = df['matter_panel_duration'].dt.total_seconds() / 60
df['life_panel_duration'] = df['life_panel_duration'].dt.total_seconds() / 60
df['thought_panel_duration'] = df['thought_panel_duration'].dt.total_seconds() / 60

In [54]:
df.to_csv('et_data.csv')

In [55]:
df.columns

Index(['recording id', 'wearer_name', 'has_corruption', 'door_sequence',
       'matter_duration', 'life_duration', 'thought_duration', 'p1_duration',
       'p2_duration', 'p3_duration', 'p4_duration', 'p5_duration',
       'p6_duration', 'p7_duration', 'phone_pickups', 'phone_duration',
       'overall_duration', 'weekday', 'overall_panel_duration',
       'number_of_panels', 'matter_panel_num', 'life_panel_num',
       'thought_panel_num', 'matter_panel_duration', 'life_panel_duration',
       'thought_panel_duration'],
      dtype='object')

In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 186 entries, 0 to 185
Data columns (total 26 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   recording id            186 non-null    object 
 1   wearer_name             186 non-null    object 
 2   has_corruption          186 non-null    bool   
 3   door_sequence           180 non-null    object 
 4   matter_duration         186 non-null    float64
 5   life_duration           186 non-null    float64
 6   thought_duration        186 non-null    float64
 7   p1_duration             186 non-null    float64
 8   p2_duration             186 non-null    float64
 9   p3_duration             186 non-null    float64
 10  p4_duration             186 non-null    float64
 11  p5_duration             186 non-null    float64
 12  p6_duration             186 non-null    float64
 13  p7_duration             186 non-null    float64
 14  phone_pickups           186 non-null    fl

In [57]:
df

,recording id,wearer_name,has_corruption,door_sequence,matter_duration,life_duration,thought_duration,p1_duration,p2_duration,p3_duration,...,overall_duration,weekday,overall_panel_duration,number_of_panels,matter_panel_num,life_panel_num,thought_panel_num,matter_panel_duration,life_panel_duration,thought_panel_duration
0,b5cff087-0b53-499e-997d-05af39002d55,D01-P010,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out",1.964444,1.239954,12.542427,0.000000,0.000000,0.000000,...,20.165117,Tuesday,0.000000,0,0,0,0,0.000000,0.000000,0.000000
1,0ca2b022-aee7-4299-a2a1-f70663e48628,D03-P011,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d6.out",1.638814,3.040312,7.711363,0.000000,0.000000,0.000000,...,17.313833,Tuesday,2.340924,2,0,2,0,0.000000,2.340924,0.000000
2,5459d7cb-ad76-41b5-8547-78d0993ce78b,D03-P012,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out",2.726877,6.574081,2.205369,0.000000,0.000000,0.000000,...,16.004833,Tuesday,0.000000,0,0,0,0,0.000000,0.000000,0.000000
3,d954354a-0daf-434e-b34d-fba86285e06d,D01-P011,False,"d1.in,d2.out,d3.in,d3.out,d3.in,d4.out,d6.in,d...",1.355124,1.510922,6.546361,4.273640,1.180700,0.000000,...,19.887383,Tuesday,5.594569,3,1,1,0,1.180700,0.140229,0.000000
4,b726709d-b664-48c0-8161-0b5b8533d84b,D05-P013,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d5.out",1.413269,5.471537,5.203532,0.603050,0.000000,0.000000,...,17.079667,Tuesday,1.523487,2,0,1,0,0.000000,0.920437,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181,40548af0-64d4-4e4c-baa8-f0f8c76ac823,D03-P066,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out",0.752095,2.601578,10.001148,0.000000,0.000000,0.000000,...,14.827333,Sunday,0.000000,0,0,0,0,0.000000,0.000000,0.000000
182,a338fabd-600c-4be0-88a4-a00b19e3b487,D01-P063,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out",2.570361,6.486856,5.900128,0.000000,1.464672,0.000000,...,21.812683,Sunday,2.071402,2,1,0,1,1.464672,0.000000,0.606730
183,8de10704-68d9-4bdd-957c-42a60fdbf562,D05-P050,False,"d1.in,d2.out,d3.in,d4.out,d5.in,d6.out",1.661338,4.743748,19.370472,0.000000,0.000000,0.733121,...,28.294667,Sunday,0.733121,1,1,0,0,0.733121,0.000000,0.000000
184,6c8a0694-0c8f-4ae3-978e-e3657f3f4fd1,D02-P058,False,"d1.in,d2.out,d3.in,d4.out,d6.in,d6.out",1.545154,5.426247,8.577607,0.000000,0.000000,0.372478,...,23.148350,Sunday,5.711166,3,1,1,1,0.372478,5.225336,0.113352
